**1. Importación de librerías**

En esta celda se importan las librerías necesarias para realizar la detección y el reconocimiento de matrículas, así como para procesar las imágenes y almacenar los resultados.

In [ ]:
import cv2
import csv
import math 
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import json
import random
import shutil
from pathlib import Path
import torch
import easyocr
import time
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

**2. Reconocimiento de matrículas mediante EasyOCR en vídeo**

Se procesan los fotogramas del vídeo utilizando YOLO para detectar personas, bicicletas, coches y matrículas. Posteriormente, se aplica EasyOCR sobre las matrículas detectadas para reconocer sus caracteres y asociarlos a los vehículos correspondientes.

In [ ]:

#Se cargan los modelos para detectar objetos y matrículas
model = YOLO('yolo11n.pt')  # Personas, bicicletas y coches
model_matriculas = YOLO("../../matriculas/runs/train_val_matriculas/weights/best.pt")

#Se inicializa EasyOCR para reconocer el texto de las matrículas
reader = easyocr.Reader(['es'], gpu=True)

#Se definen las etiquetas de las distintas clases
classNames = ["person", "bicycle", "car"]

#Se abre el vídeo que se va a procesar
vid = cv2.VideoCapture("../P4/C0142.mp4")

track_history = defaultdict(lambda: [])

#Se obtienen las características del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

#Se crea el vídeo de salida para almacenar los fotogramas procesados
out = cv2.VideoWriter("C0142_procesado_Ocr.mp4", fourcc, fps, (width, height))

#Se crea el archivo CSV y se definen los campos para almacenar los resultados
archivo_csv = open("C0142_resultados_Ocr.csv", "w", newline="", encoding="utf-8")
campos = [
    "fotograma", "tipo_objeto", "confianza",
    "identificador_tracking", "x1", "y1", "x2", "y2",
    "matricula_en_su_caso", "confianza_matricula",
    "mx1", "my1", "mx2", "my2", "texto_matricula"
]

escritor = csv.DictWriter(archivo_csv, fieldnames=campos)
escritor.writeheader()

#Se inicializan los contadores y los conjuntos para almacenar los IDs contabilizados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

#Se inicializa el contador de fotogramas
num_fotograma = 0

while(True):

    #Se obtiene cada fotograma del vídeo
    ret, img = vid.read()

    #Se comprueba que el fotograma se haya leído correctamente
    if ret:

        num_fotograma += 1

        #Se realiza el seguimiento de personas, bicicletas y coches
        results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

        #Se detectan las matrículas presentes en el fotograma
        results_matriculas = model_matriculas.predict(img, conf=0.25, verbose=False)

        #Se inicializa la lista para almacenar las matrículas detectadas
        matriculas = []

        for r_mat in results_matriculas:
            for box_mat in r_mat.boxes:

                #Se obtienen las coordenadas de cada matrícula detectada
                mx1, my1, mx2, my2 = box_mat.xyxy[0].cpu().numpy()
                mx1, my1, mx2, my2 = int(mx1), int(my1), int(mx2), int(my2)

                #Se ajustan las coordenadas para evitar que salgan de los límites de la imagen
                mx1 = max(0, mx1)
                my1 = max(0, my1)
                mx2 = min(width, mx2)
                my2 = min(height, my2)

                #Se obtiene la confianza de la detección de la matrícula
                confianza_mat = float(box_mat.conf[0])

                #Se recorta la región de la imagen correspondiente a la matrícula
                roi_mat = img[my1:my2, mx1:mx2]

                #Se inicializan las variables para almacenar el texto y la confianza del reconocimiento
                texto_mat = ""
                confianza_ocr = 0.0

                #Se comprueba que la región recortada contiene información
                if roi_mat.size > 0:

                    #Se convierte la imagen a escala de grises
                    roi_mat = cv2.cvtColor(roi_mat, cv2.COLOR_BGR2GRAY)

                    #Se amplía la imagen 5 veces mediante interpolación cúbica
                    roi_mat = cv2.resize(roi_mat, None, fx=5, fy=5, interpolation=cv2.INTER_CUBIC)

                    #Se mejora el contraste de la imagen mediante CLAHE
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    roi_mat = clahe.apply(roi_mat)

                    resultado_ocr = reader.readtext(roi_mat, allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ')

                    #Se selecciona el resultado con mayor confianza de reconocimiento
                    if len(resultado_ocr) > 0:
                        mejor_resultado = max(resultado_ocr, key=lambda x: x[2])
                        texto_mat = mejor_resultado[1].replace(" ", "").upper()
                        confianza_ocr = float(mejor_resultado[2])

                #Se almacena la información obtenida de la matrícula
                matriculas.append({
                    "x1": mx1, "y1": my1, "x2": mx2, "y2": my2,
                    "confianza": confianza_mat, "confianza_ocr": confianza_ocr,
                    "texto": texto_mat
                })

        #Se procesan las detecciones de personas, bicicletas y coches
        for r in results:

            boxes = r.boxes

            for box in boxes:

                #Se obtienen las coordenadas de la caja delimitadora de cada objeto
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                #Se ajustan las coordenadas para evitar que salgan de los límites de la imagen
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                #Se obtiene la confianza de la detección
                confidence = float(box.conf[0])
                print("Confianza =", round(confidence, 2))

                #Se identifica la clase del objeto detectado
                cls = int(box.cls[0])
                nombre_clase = classNames[cls]
                print("Clase =", nombre_clase)

                #Se obtiene el identificador de seguimiento del objeto
                if box.id is not None:
                    track_id = str(int(box.id[0]))
                else:
                    track_id = ""

                #Se inicializa la variable para almacenar la matrícula asociada
                matricula_asociada = None

                #Se buscan matrículas asociadas únicamente a los coches
                if cls == 2:

                    for mat in matriculas:

                        #Se calcula el centro de la caja delimitadora de la matrícula
                        centro_x = (mat["x1"] + mat["x2"]) // 2
                        centro_y = (mat["y1"] + mat["y2"]) // 2

                        #Se comprueba si el centro de la matrícula se encuentra dentro del coche
                        if (x1 <= centro_x <= x2 and
                            y1 <= centro_y <= y2):

                            #Se selecciona la matrícula con mayor confianza de detección
                            if (matricula_asociada is None or
                                mat["confianza"] > matricula_asociada["confianza"]):
                                matricula_asociada = mat

                #Se actualizan los contadores si el ID todavía no ha sido contabilizado
                if box.id is not None:

                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                #Se obtiene la información de la matrícula para almacenarla en el CSV
                if matricula_asociada is not None:
                    tiene_matricula = "Si"
                    confianza_matricula = matricula_asociada["confianza"]
                    mx1 = matricula_asociada["x1"]
                    my1 = matricula_asociada["y1"]
                    mx2 = matricula_asociada["x2"]
                    my2 = matricula_asociada["y2"]
                    texto_matricula = matricula_asociada["texto"]

                else:
                    tiene_matricula = "No"
                    confianza_matricula = ""
                    mx1 = ""
                    my1 = ""
                    mx2 = ""
                    my2 = ""
                    texto_matricula = ""

                #Se escribe la información de cada objeto detectado en el archivo CSV
                escritor.writerow({
                    "fotograma": num_fotograma, "tipo_objeto": nombre_clase,
                    "confianza": confidence, "identificador_tracking": track_id,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "matricula_en_su_caso": tiene_matricula,
                    "confianza_matricula": confianza_matricula,
                    "mx1": mx1, "my1": my1, "mx2": mx2, "my2": my2,
                    "texto_matricula": texto_matricula
                })

                #Se asigna un color diferente a cada clase de objeto
                if cls == 0:
                    R = 128
                    G = 0
                    B = 0
                elif cls == 1:
                    R = 0
                    G = 128
                    B = 0
                elif cls == 2:
                    R = 0
                    G = 0
                    B = 128

                #Se dibujan la caja delimitadora y la etiqueta de seguimiento del objeto
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + nombre_clase, [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

                #Se muestra la información de la matrícula cuando se encuentra asociada a un coche
                if matricula_asociada is not None:

                    texto = texto_matricula

                    #Se indica cuando no se ha podido reconocer el texto de la matrícula
                    if texto == "":
                        texto = "Matricula no reconocida"

                    #Se dibujan la caja delimitadora y el texto reconocido de la matrícula
                    cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 255), 2)
                    cv2.putText(img, texto, (mx1, max(20, my1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        #Se muestran los contadores de personas, bicicletas y coches en el fotograma
        cv2.putText(img, f'Personas: {contadores["person"]}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.putText(img, f'Coches: {contadores["car"]}',
                    (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        #Se guarda el fotograma procesado en el vídeo de salida
        out.write(img)

        #Se muestra el fotograma procesado
        cv2.imshow('Vid', img)

    else:
        break

    #Se detiene la ejecución al pulsar ESC
    if cv2.waitKey(20) == 27:
        break

#Se libera el objeto de captura del vídeo
vid.release()

#Se libera el objeto de escritura del vídeo
out.release()

archivo_csv.close()

cv2.destroyAllWindows()

**3. Preparación del conjunto de imágenes de matrículas**

Se generan recortes individuales de las matrículas a partir de las imágenes originales y sus anotaciones JSON. También se conserva el texto real de cada matrícula para utilizarlo posteriormente como referencia durante la evaluación de los sistemas OCR.

In [ ]:
#Se obtiene el directorio de trabajo actual
carpeta_actual = Path.cwd()

#Se busca la raiz del proyecto comprobando los directorios superiores
carpeta_proyecto = next(
    (p for p in [carpeta_actual, *carpeta_actual.parents]
     if (p / "matriculas" / "images").exists()
     and (p / "matriculas" / "annotations").exists()),
    None
)

#Se comprueba que se haya encontrado la carpeta del proyecto
if carpeta_proyecto is None:
    raise FileNotFoundError("No se encuentra la carpeta matriculas")

#Se definen las rutas de las imagenes, anotaciones y recortes
carpeta_imagenes = carpeta_proyecto / "matriculas" / "images"
carpeta_anotaciones = carpeta_proyecto / "matriculas" / "annotations"
carpeta_p4b = carpeta_proyecto / "P4b"
carpeta_recortes = carpeta_p4b / "images_recortadas"

#Se crea la carpeta de recortes si no existe
carpeta_recortes.mkdir(parents=True, exist_ok=True)

#Se localizan las imagenes con extension JPG o JPEG
extensiones = [".jpg", ".jpeg"]
imagenes = [
    p for p in carpeta_imagenes.iterdir()
    if p.suffix.lower() in extensiones
]

#Se inicializan las listas y contadores de resultados
resultados = []
total_sin_json = 0
total_sin_matricula = 0
total_error_imagen = 0

#Se establece un margen adicional del 8% alrededor de cada matricula
margen = 0.08

#Se procesan las imagenes encontradas
for imagen_path in imagenes:

    #Se localiza el archivo JSON correspondiente a la imagen
    archivo_json = carpeta_anotaciones / (imagen_path.stem + ".json")

    #Se omiten las imagenes que no tienen un archivo de anotaciones
    if not archivo_json.exists():
        total_sin_json += 1
        continue

    #Se carga la imagen mediante un metodo compatible con rutas Unicode
    img = cv2.imdecode(np.fromfile(str(imagen_path), dtype=np.uint8), cv2.IMREAD_COLOR)

    #Se contabilizan las imagenes que no se han podido cargar
    if img is None:
        total_error_imagen += 1
        continue

    #Se cargan las anotaciones del archivo JSON
    with open(archivo_json, "r", encoding="utf-8") as f:
        anotaciones = json.load(f)

    #Se obtiene el alto y el ancho de la imagen
    alto, ancho = img.shape[:2]

    #Se procesa cada anotacion de la imagen
    for i, anotacion in enumerate(anotaciones):

        #Se obtiene el texto de la matricula y se normaliza
        matricula = str(anotacion.get("plate", "")).strip().upper()

        #Se omiten las anotaciones que no contienen una matricula valida
        if not matricula or matricula == "NONE":
            total_sin_matricula += 1
            continue

        #Se obtienen las coordenadas normalizadas del rectangulo de la matricula
        cx, cy, w, h = anotacion["rectangle"]

        #Se convierten las coordenadas a pixeles y se añade el margen
        x1 = int((cx - w / 2 - w * margen) * ancho)
        y1 = int((cy - h / 2 - h * margen) * alto)
        x2 = int((cx + w / 2 + w * margen) * ancho)
        y2 = int((cy + h / 2 + h * margen) * alto)

        #Se ajustan las coordenadas para que permanezcan dentro de la imagen
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(ancho, x2)
        y2 = min(alto, y2)

        #Se obtiene el recorte correspondiente a la matricula
        recorte = img[y1:y2, x1:x2]

        #Se descartan los recortes vacios
        if recorte.size == 0:
            continue

        #Se define el nombre y la ruta del archivo de salida
        nombre = f"{imagen_path.stem}_{i}.jpg"
        ruta_salida = carpeta_recortes / nombre

        #Se codifica el recorte en formato JPG para guardarlo
        correcto, buffer = cv2.imencode(".jpg", recorte)

        #Se guarda el recorte si la codificacion se ha realizado correctamente
        if correcto:
            buffer.tofile(str(ruta_salida))

            #Se registra el recorte generado y su matricula real
            if ruta_salida.exists():
                resultados.append({
                    "imagen": nombre,
                    "matricula_real": matricula,
                    "imagen_original": imagen_path.name
                })

#Se guarda un CSV con los recortes y sus matriculas reales
ruta_csv = carpeta_p4b / "matriculas_reales.csv"

with open(ruta_csv, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=["imagen", "matricula_real", "imagen_original"])
    escritor.writeheader()
    escritor.writerows(resultados)

#Se muestra un resumen de la preparacion del dataset
print("DATASET DE MATRICULAS PREPARADO")
print("--------------------------------")
print("Imagenes originales:", len(imagenes))
print("Recortes generados:", len(resultados))
print("Imagenes sin JSON:", total_sin_json)
print("Anotaciones sin matricula:", total_sin_matricula)
print("Errores al cargar imagenes:", total_error_imagen)
print("--------------------------------")
print("Carpeta:", carpeta_recortes)
print("CSV:", ruta_csv)

**4. Evaluación del reconocimiento mediante EasyOCR**

Se evalúa EasyOCR utilizando las imágenes recortadas y las matrículas reales como referencia. Se comparan las predicciones con los textos originales para calcular la exactitud del reconocimiento y medir el tiempo de procesamiento.

In [ ]:
#Se definen las rutas de la carpeta de recortes y del archivo CSV con las etiquetas reales
carpeta_recortes = Path.cwd() / "images_recortadas"
ruta_csv = Path.cwd() / "matriculas_reales.csv"

#Se cargan las etiquetas reales desde el archivo CSV
with open(ruta_csv, "r", encoding="utf-8") as f:
    datos = list(csv.DictReader(f))

#Se seleccionan únicamente las imagenes que existen en la carpeta de recortes
datos_disponibles = [
    fila for fila in datos
    if (carpeta_recortes / fila["imagen"]).is_file()
]

#Se comprueba si hay una GPU disponible y se inicializa EasyOCR
usar_gpu = torch.cuda.is_available()
reader = easyocr.Reader(["en"], gpu=usar_gpu)

#Se muestra la informacion inicial del procesamiento
print("GPU disponible:", usar_gpu)
print("Imagenes en el CSV:", len(datos))
print("Imagenes disponibles:", len(datos_disponibles))
print("--------------------------------")

#Se inicializan la lista de resultados y el tiempo total de procesamiento
resultados = []
tiempo_total = 0

#Se procesan las imagenes disponibles
for i, fila in enumerate(datos_disponibles):

    #Se obtiene la ruta de la imagen actual
    ruta = carpeta_recortes / fila["imagen"]

    #Se carga la imagen mediante un metodo compatible con rutas Unicode
    imagen = cv2.imdecode(np.fromfile(str(ruta), dtype=np.uint8), cv2.IMREAD_COLOR)

    #Se omiten las imagenes que no se han podido cargar
    if imagen is None:
        print("No se pudo cargar:", ruta.name)
        continue

    #Se mide el tiempo empleado por EasyOCR para reconocer la matricula
    inicio = time.time()

    lectura = reader.readtext(imagen, allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789", detail=1)

    tiempo = time.time() - inicio
    tiempo_total += tiempo

    #Se unen los textos reconocidos y se obtiene la matricula real
    prediccion = "".join(r[1] for r in lectura)
    real = fila["matricula_real"].upper()

    #Se almacenan la prediccion, su comparacion con la matricula real y el tiempo empleado
    resultados.append({"imagen": fila["imagen"],
        "real": real,
        "prediccion": prediccion,
        "correcta": prediccion == real,
        "tiempo": tiempo
    })

    #Se muestra el resultado del reconocimiento de cada imagen
    print(f"{i + 1}/{len(datos_disponibles)} | Real: {real} | OCR: {prediccion}")

#Se calculan el numero total de resultados y las lecturas exactas
total = len(resultados)
correctas = sum(r["correcta"] for r in resultados)

#Se muestran las metricas obtenidas si se han procesado imagenes
if total > 0:
    print("\nRESULTADOS DE EASYOCR")
    print("--------------------------------")
    print("Imagenes procesadas:", total)
    print("Lecturas exactas:", correctas)
    print("Lecturas incorrectas:", total - correctas)
    print("Exactitud:", round(correctas / total * 100, 2), "%")
    print("Tiempo total:", round(tiempo_total, 2), "segundos")
    print("Tiempo medio:", round(tiempo_total / total * 1000, 2), "ms")

#Se define la ruta del archivo CSV donde se guardaran las predicciones
ruta_resultados = Path.cwd() / "resultados_easyocr.csv"

#Se guardan los resultados de las predicciones en un archivo CSV
with open(ruta_resultados, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=["imagen", "real", "prediccion", "correcta", "tiempo"])
    escritor.writeheader()
    escritor.writerows(resultados)

#Se muestra la ruta del archivo con los resultados
print("\nResultados guardados en:", ruta_resultados)

**5. Evaluación del reconocimiento mediante SmolVLM**

Se utiliza SmolVLM para reconocer el texto de las matrículas a partir de las mismas imágenes empleadas en la evaluación anterior. Se almacenan las predicciones y los tiempos de procesamiento para poder comparar posteriormente los resultados de ambos sistemas.

In [ ]:
#Se carga el modelo SmolVLM y se selecciona el dispositivo de procesamiento
modelo = "HuggingFaceTB/SmolVLM-256M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

#Se inicializa el procesador del modelo
processor = AutoProcessor.from_pretrained(modelo)

#Se carga el modelo utilizando precision reducida si se dispone de GPU
model = AutoModelForVision2Seq.from_pretrained(modelo, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)

#Se establece el modelo en modo evaluacion
model.eval()

#Se definen las rutas de la carpeta de recortes y del archivo CSV con las matriculas reales
carpeta = Path.cwd() / "images_recortadas"
archivo_csv = Path.cwd() / "matriculas_reales.csv"

#Se cargan las matriculas reales desde el archivo CSV
with open(archivo_csv, encoding="utf-8-sig") as f:
    datos = list(csv.DictReader(f))

#Se seleccionan unicamente las imagenes que existen en la carpeta de recortes
datos = [d for d in datos if (carpeta / d["imagen"]).is_file()]

#Se inicializa la lista donde se almacenaran los resultados
resultados = []

#Se define la instruccion que se enviara al modelo para reconocer las matriculas
mensajes = [{
    "role": "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": "Read the license plate. Return only the letters and numbers."}
    ]
}]

#Se prepara el formato de entrada que espera el modelo
texto = processor.apply_chat_template(mensajes, add_generation_prompt=True)

#Se procesan las imagenes disponibles
for i, fila in enumerate(datos):

    #Se carga la imagen y se convierte al formato RGB
    imagen = Image.open(carpeta / fila["imagen"]).convert("RGB")

    #Se prepara la imagen y el texto para introducirlos en el modelo
    entradas = processor(text=texto, images=[imagen], return_tensors="pt").to(device)

    #Se mide el tiempo empleado por el modelo para generar la prediccion
    inicio = time.time()

    with torch.inference_mode():
        salida = model.generate(**entradas, max_new_tokens=20)

    tiempo = time.time() - inicio

    #Se extraen los tokens generados por el modelo y se obtiene el texto reconocido
    tokens = salida[:, entradas["input_ids"].shape[1]:]
    prediccion = processor.batch_decode(tokens, skip_special_tokens=True)[0].strip()

    #Se almacenan la imagen, la matricula real, la prediccion y el tiempo empleado
    resultados.append({
        "imagen": fila["imagen"],
        "matricula_real": fila["matricula_real"],
        "prediccion": prediccion,
        "tiempo": round(tiempo, 4)
    })

    #Se cierra la imagen despues de procesarla
    imagen.close()

    #Se muestra el progreso cada 25 imagenes procesadas
    if (i + 1) % 25 == 0:
        print(f"Procesadas {i + 1}/{len(datos)}")

#Se guardan las predicciones y los tiempos de procesamiento en un archivo CSV
with open("resultados_smolvlm.csv", "w", encoding="utf-8-sig", newline="") as f:
    escritor = csv.DictWriter(f, fieldnames=resultados[0].keys())
    escritor.writeheader()
    escritor.writerows(resultados)

#Se muestra un resumen del procesamiento realizado
print("\nEvaluacion terminada.")
print("Imagenes procesadas:", len(resultados))
print("Resultados guardados en resultados_smolvlm.csv")